# 🚀 Finetune Financial Text-to-SQL trên Kaggle (GPU 2x T4 / P100)

Notebook này được thiết kế chuyên biệt để finetune mô hình **Qwen3.5-4B** (chuẩn cấu hình từ FinetuneV1) với tập dữ liệu **Financial Text-to-SQL Gold Standard** (646 mẫu có suy luận `<think>`, Pure ANSI SQL và công thức toán học).

### Tính năng nổi bật:
1. **Tăng tốc cực hạn với Unsloth**: Tiết kiệm 80% VRAM, tốc độ x2-x5 so với HuggingFace tiêu chuẩn.
2. **Response-only Loss**: Chỉ tính hàm mất mát trên câu trả lời của mô hình (`<think>` + SQL), không tính trên câu hỏi người dùng.
3. **Merge Model trực tiếp (16-bit Full Model)**: Gộp toàn bộ trọng số LoRA vào Base Model thành model độc lập sẵn sàng cho vLLM, SGLang, Transformers.
4. **Tự động Push lên Hugging Face**: Tự động login và đẩy cả bản LoRA, Merged 16-bit và GGUF (cho Ollama) lên Hugging Face Hub.

⏱️ **Thời gian huấn luyện dự kiến**: ~4 đến 6 phút trên Kaggle GPU T4.

In [ ]:
# 1. Kiểm tra cấu hình GPU Kaggle
import os, sys
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ LỖI NGHIÊM TRỌNG: Bạn chưa bật GPU trên Kaggle!\n" 
                       "👉 Hướng dẫn khắc phục:\n" 
                       "1. Nhìn sang cột bên phải (Notebook Settings)\n" 
                       "2. Tìm mục 'Accelerator'\n" 
                       "3. Chọn 'GPU T4 x2' (hoặc 'GPU P100')\n" 
                       "4. Bật 'Internet on'\n" 
                       "5. Sau đó bấm Chạy lại (Run All)!")

!nvidia-smi

# 2. Cài đặt Unsloth, unsloth_zoo và các thư viện cần thiết
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"

# Cài đặt Unsloth & unsloth_zoo tối ưu cho môi trường Kaggle
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets pyyaml huggingface_hub

print("\n>>> Đã kiểm tra GPU và cài đặt môi trường Unsloth thành công!")

In [ ]:
# ====================================================================
# 2. CẤU HÌNH NGƯỜI DÙNG & HUGGING FACE TOKEN (ĐIỀN VÀO ĐÂY)
# ====================================================================

# Điền Hugging Face Write Token của bạn (Tạo tại https://huggingface.co/settings/tokens)
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # <-- THAY BẰNG TOKEN CỦA BẠN

# Tên Repo bạn muốn tạo trên Hugging Face
HF_USERNAME = "your_username"  # <-- THAY BẰNG USERNAME HUGGING FACE CỦA BẠN
HF_REPO_MERGED = f"{HF_USERNAME}/Qwen3.5-4B-Financial-SQL"        # Model đầy đủ 16-bit
HF_REPO_LORA   = f"{HF_USERNAME}/Qwen3.5-4B-Financial-SQL-LoRA"   # LoRA Adapter

# Base Model chuẩn theo thiết kế FinetuneV1: Qwen3.5-4B
BASE_MODEL_NAME = "Qwen/Qwen3.5-4B"

# Tự động tìm file train.jsonl và val.jsonl trong Kaggle Input hoặc thư mục hiện tại
import glob
data_candidates = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
if data_candidates:
    TRAIN_FILE = data_candidates[0]
    VAL_FILE = TRAIN_FILE.replace("train.jsonl", "val.jsonl")
else:
    TRAIN_FILE = "train.jsonl"
    VAL_FILE = "val.jsonl"

print(f"Dataset Train: {TRAIN_FILE}")
print(f"Dataset Val:   {VAL_FILE}")

In [ ]:
# ====================================================================
# 3. TẢI BASE MODEL & GẮN ADAPTER LORA (QLORA 4-BIT)
# ====================================================================
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None        # Tự động chọn Float16 hoặc Bfloat16
load_in_4bit = True # 4-bit quantization (Chỉ tốn ~5-6GB VRAM!)

print(f">>> Đang tải base model: {BASE_MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Thiết lập LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print(">>> Khởi tạo Base Model và LoRA Adapter thành công!")

In [ ]:
# ====================================================================
# 4. NẠP DỮ LIỆU & ĐỊNH DẠNG CHATML
# ====================================================================
from datasets import load_dataset

dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "val": VAL_FILE})
print(f"Số mẫu Train: {len(dataset['train']):,} | Số mẫu Val: {len(dataset['val']):,}")

# Định dạng từng mẫu thành văn bản ChatML bằng tokenizer chuẩn
def format_prompts(batch):
    texts = [tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False) for conv in batch["messages"]]
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

print("\n--- MẪU DỮ LIỆU ĐÃ ĐỊNH DẠNG (SAMPLE 0) ---")
print(dataset["train"][0]["text"][:600] + "\n...")

In [ ]:
# ====================================================================
# 5. CẤU HÌNH BỘ HUẤN LUYỆN (SFTTRAINER & RESPONSE-ONLY LOSS)
# ====================================================================
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only

training_args = TrainingArguments(
    output_dir="./outputs/qwen_financial_sql",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 2 * 4 = 8
    warmup_ratio=0.05,
    learning_rate=2e-4,
    num_train_epochs=3,             # 3 epochs (~218 steps)
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=training_args,
)

# CHỈ TÍNH LOSS TRÊN PHẦN TRẢ LỜI CỦA ASSISTANT (<think> + SQL)
# Không tính loss trên câu hỏi hoặc prompt hệ thống
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print(">>> Bộ huấn luyện SFTTrainer đã sẵn sàng!")

In [ ]:
# ====================================================================
# 6. BẮT ĐẦU HUẤN LUYỆN (TRAINING)
# ====================================================================
print(">>> BẮT ĐẦU FINETUNING TRÊN KAGGLE GPU...")
trainer_stats = trainer.train()
print(">>> HUẤN LUYỆN HOÀN TẤT!")
print(trainer_stats)

In [ ]:
# ====================================================================
# 7. KIỂM THỬ INFERENCE TRỰC TIẾP TRÊN MODEL VỪA TRAIN
# ====================================================================
FastLanguageModel.for_inference(model)

test_question = "Lợi nhuận sau thuế của CTCP Chứng khoán FPT năm 2023 là bao nhiêu tỷ đồng?"
system_prompt = dataset["train"][0]["messages"][0]["content"]

test_prompt = f"""<|im_start|>system
{system_prompt}<|im_end|>
<|im_start|>user
{test_question}<|im_end|>
<|im_start|>assistant
"""

# Tránh nhầm lẫn với Vision processor của Qwen, lấy text tokenizer chuẩn
text_tok = getattr(tokenizer, "tokenizer", tokenizer)
inputs = text_tok(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=True, temperature=0.0)

# In câu trả lời của mô hình
response = text_tok.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
print("--- CÂU TRẢ LỜI CỦA MÔ HÌNH VỪA FINETUNE ---")
print(response)


In [ ]:
# ====================================================================
# 8. MERGE MODEL (16-BIT FULL MODEL) & PUSH LÊN HUGGING FACE HUB
# ====================================================================
from huggingface_hub import login, HfApi

# 1. Lưu dự phòng bản 16-bit vào ổ cứng Kaggle trước để tuyệt đối an toàn
print(">>> Đang lưu bản Merged 16-bit vào /kaggle/working/qwen3_5_4b_merged_16bit ...")
model.save_pretrained_merged("/kaggle/working/qwen3_5_4b_merged_16bit", tokenizer, save_method="merged_16bit")
print(">>> Đã lưu an toàn vào ổ cứng Kaggle!")

# 2. Đăng nhập Hugging Face
print("\n>>> Vui lòng đăng nhập Hugging Face (Token quyền Write):")
if 'HF_TOKEN' in locals() and HF_TOKEN and not HF_TOKEN.startswith("hf_xxx"):
    try:
        login(token=HF_TOKEN)
    except Exception:
        login()
else:
    login()

# 3. Tự động lấy Username chính xác từ Hugging Face
api = HfApi()
user_info = api.whoami()
username = user_info["name"]
print(f"\n>>> Đăng nhập thành công với tài khoản: {username}!")

repo_merged = f"{username}/Qwen3.5-4B-Financial-SQL"
repo_lora   = f"{username}/Qwen3.5-4B-Financial-SQL-LoRA"

# 4. Đẩy LoRA Adapter (~100MB)
print(f"\n[1/2] Đang đẩy LoRA Adapter lên: {repo_lora} ...")
model.push_to_hub(repo_lora, tokenizer=tokenizer)

# 5. Đẩy Full Merged 16-bit Model (Upload trực tiếp thư mục đã merge)
print(f"\n[2/2] Đang đẩy Merged 16-bit Model lên: {repo_merged} ...")
api.create_repo(repo_id=repo_merged, exist_ok=True)
api.upload_folder(
    folder_path="/kaggle/working/qwen3_5_4b_merged_16bit",
    repo_id=repo_merged,
    repo_type="model"
)
print(f"\n🎉🎉🎉 HOÀN THÀNH XUẤT SẮC! Model đã có mặt tại: https://huggingface.co/{repo_merged}")
